<a href="https://colab.research.google.com/github/MUKUL-PRASAD-SIGH/ML_Colab/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

texts = [
    "Win money now", "Hello how are you",
    "Claim your free prize", "Let's meet tomorrow",
    "Cheap loans available", "Are you coming today",
    "Win a free iPhone now", "Claim your free prize today",
    "Congratulations you won lottery", "Limited offer buy now",
    "Cheap loans available", "Earn money from home",
    "Click here to get reward", "You are selected for cash prize",
    "Lowest price on meds", "Free entry in contest",
    "Hey are you coming today", "Let's meet for lunch",
    "Can you call me later", "How was your day",
    "I will be late today", "See you tomorrow",
    "Don't forget the meeting", "Happy birthday have fun",
    "Are we still on for tonight", "Please send the report"
]
labels = [1,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0]  # 1 = spam, 0 = not spam

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.625


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = pd.read_csv("spam_test.csv")

texts = data["text"]
labels = data["label"]

# Convert text → TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.3, random_state=42, stratify=labels
)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


Accuracy: 0.5
              precision    recall  f1-score   support

         ham       0.50      0.33      0.40         3
        spam       0.50      0.67      0.57         3

    accuracy                           0.50         6
   macro avg       0.50      0.50      0.49         6
weighted avg       0.50      0.50      0.49         6



# New Section

In [ ]:
from gensim.models import Word2Vec
import torch
import torch.nn as nn
import torch.optim as optim

# ------------------------
# 1. Data
# ------------------------
sentences = [
    ["i","love","nlp"],
    ["nlp","is","fun"],
    ["i","love","machine","learning"],
    ["machine","learning","is","powerful"]
]

# labels: 0 = NLP, 1 = ML
labels = torch.tensor([0, 0, 1, 1])

# ------------------------
# 2. Train Word2Vec
# ------------------------
w2v = Word2Vec(sentences, vector_size=20, min_count=1)

# convert sentence to vectors
def sentence_to_vec(sent):
    return torch.tensor([w2v.wv[w] for w in sent], dtype=torch.float)

X = [sentence_to_vec(s) for s in sentences]

# ------------------------
# 3. Pad sequences (important!)
# ------------------------
X_padded = nn.utils.rnn.pad_sequence(X, batch_first=True)

# ------------------------
# 4. LSTM Model
# ------------------------
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(20, 16, batch_first=True)
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]      # last timestep
        return self.fc(out)

model = LSTMModel()

# ------------------------
# 5. Loss & Optimizer
# ------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ------------------------
# 6. Training loop
# ------------------------
for epoch in range(50):
    outputs = model(X_padded)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# ------------------------
# 7. Test
# ------------------------
with torch.no_grad():
    preds = torch.argmax(model(X_padded), dim=1)

print("True labels:", labels.tolist())
print("Predicted  :", preds.tolist())


/tmp/ipython-input-2592621484.py:26: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return torch.tensor([w2v.wv[w] for w in sent], dtype=torch.float)


Epoch 0, Loss: 0.6977
Epoch 10, Loss: 0.6799
Epoch 20, Loss: 0.5758
Epoch 30, Loss: 0.1773
Epoch 40, Loss: 0.0523
True labels: [0, 0, 1, 1]
Predicted  : [0, 0, 1, 1]


In [ ]:
!pip3 install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.3 MB/s eta 0:00:00


In [ ]:
# ------------------------
# 8. Test on NEW sentence
# ------------------------
def predict_sentence(sentence):
    words = sentence.lower().split()

    # convert to word2vec vectors (only known words)
    vecs = [w2v.wv[w] for w in words if w in w2v.wv]

    if len(vecs) == 0:
        print("No known words in sentence!")
        return

    x = torch.tensor(vecs, dtype=torch.float).unsqueeze(0)  # shape: (1, seq_len, 20)

    with torch.no_grad():
        output = model(x)
        pred = torch.argmax(output, dim=1).item()

    if pred == 0:
        print(f"Sentence: '{sentence}' → Class: NLP (0)")
    else:
        print(f"Sentence: '{sentence}' → Class: ML (1)")


# Try new sentences
predict_sentence("nlp is powerful")
predict_sentence("i love ml ")
predict_sentence("machine learning is fun")


Sentence: 'nlp is powerful' → Class: NLP (0)
Sentence: 'i love ml ' → Class: NLP (0)
Sentence: 'machine learning is fun' → Class: ML (1)


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip


--2026-02-01 18:50:06--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-02-01 18:50:06--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-02-01 18:50:07--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
import numpy as np

def load_glove(path):
    embeddings = {}
    with open(path, encoding="utf8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype="float32")
            embeddings[word] = vector
    return embeddings

glove = load_glove("glove.6B.50d.txt")
print("Loaded words:", len(glove))


Loaded words: 400000


In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def most_similar(word, topn=5):
    if word not in glove:
        print("Word not in GloVe vocabulary!")
        return

    vec = glove[word]
    sims = []

    for w, v in glove.items():
        sims.append((w, cosine_sim(vec, v)))

    sims = sorted(sims, key=lambda x: x[1], reverse=True)
    return sims[1:topn+1]  # skip itself


In [ ]:
while True:
    w = input("Enter a word (or 'exit'): ").lower()
    if w == "exit":
        break
    print(most_similar(w, 5))


[('cool', np.float32(0.8605021)), ('cold', np.float32(0.8010528)), ('mix', np.float32(0.7706386)), ('dry', np.float32(0.7561463)), ('soft', np.float32(0.7462474))]
[('boy', np.float32(0.9327199)), ('woman', np.float32(0.9065281)), ('mother', np.float32(0.83476734)), ('girls', np.float32(0.8182201)), ('girlfriend', np.float32(0.8109235))]
[('ho', np.float32(0.72917265)), ('ai', np.float32(0.701225)), ('otlk', np.float32(0.665361)), ('tu', np.float32(0.659475)), ('se', np.float32(0.62460554))]
[('goodbye', np.float32(0.8537959)), ('hey', np.float32(0.8074297)), ('!', np.float32(0.79513884)), ('kiss', np.float32(0.7892293)), ('wow', np.float32(0.7641353))]
